In [ ]:
import shap
import numpy as np
import tensorflow as tf
from utils.data_loader import load_as_maps
from joblib import load

In [ ]:
path = ""

# load model
model = tf.keras.models.load_model(path + "/model.keras")

# load and prepare data
X_test, Y_test = load_as_maps(start_year=1958, end_year=1958, datasets=["exp1"],target_index=3)
map_mask = X_test[0,:,:, 10] == 1

scaler = load(path + '/scaler.pkl')
n_samples, h, w, n_features = X_test.shape
X_test_flat = X_test.reshape(-1,n_features)
X_test_scaled_flat = scaler.transform(X_test_flat)
X_test = X_test_scaled_flat.reshape(n_samples, h, w, n_features)

# X_test = patch_extract(X_test, patch_size=(4,4))

# run model and evalutate
predictions = model.predict(X_test)
predictions = predictions.reshape(n_samples, h, w)
pred_masked = predictions * map_mask

# pick a small background set
background = X_train[np.random.choice(X_train.shape[0], 50, replace=False)]

# choose some test samples to explain
X_explain = X_test[:10]

# SHAP explainer (GradientExplainer is usually better for U-Net)
explainer = shap.GradientExplainer(model, background)

# compute SHAP values
shap_values = explainer.shap_values(X_explain)

# shap_values: list of arrays (one per output), each shaped like input
# e.g. (10, 167, 360, 16)

In [ ]:
# average over batch (0), lat (1), lon (2)
global_importance = np.mean(np.abs(shap_values), axis=(0,1,2))

# result shape: (16,)

In [ ]:
feature_names = ["SST", "SAL", "ice_frac", "mixed_layer_depth",
                 "heat_flux_down", "water_flux_up",
                 "stress_X", "stress_Y",
                 "currents_X", "currents_Y",
                 "nav_lat", "nav_lon", "month", "year", "global_co2", "tmask"]

importance_norm = global_importance / global_importance.sum()

sorted_idx = np.argsort(importance_norm)[::-1]

for i in sorted_idx:
    print(f"{feature_names[i]}: {importance_norm[i]:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.barh([feature_names[i] for i in sorted_idx],
         importance_norm[sorted_idx])
plt.gca().invert_yaxis()
plt.xlabel("Normalized importance")
plt.title("Global Feature Relevance (SHAP)")
plt.show()